# ML-04 — Ranking Signal Analysis Data Contract

This notebook uses the full FlyRank warehouse release for a small, public-safe mid-panel slice. The goal is to define the row, prove the slice is usable, build five decision-time features, and demonstrate the leakage trap before modeling.

## 1. Contract: five plain-words answers

- **One row means:** one daily observation for one pseudonymized content item and client on one `report_date`.
- **Tables used:** `fact_content_daily_performance` for daily GSC performance and `dim_clients` only for the warehouse availability context.
- **Time window:** March 1–31, 2026, using the first half as decision-time features and the second half as the observed movement window.
- **What I rank / proxy:** I rank content review leads using five first-half search signals; the label proxy is whether normalized second-half impressions fall below 80% of first-half impressions.
- **Deliberate exclusion:** `content_hash_id` and `client_hash_id` stay out of features because they are pseudonymous identifiers used only for grouping and audit.

In [1]:
import getpass
import os

import duckdb
import pandas as pd

MONTH = "2026-03"
MONTH_START = "2026-03-01"
MIDPOINT = "2026-03-15"
MONTH_END = "2026-03-31"

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Enter your Hugging Face READ token (input is hidden): "
)
assert HF_TOKEN, "A Hugging Face READ token is required; it is never stored in this notebook."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MONTH = (
    f"read_parquet('{REL}/fact_content_daily_performance/"
    f"month={MONTH}/*.parquet')"
)
CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print(f"Warehouse slice: fact_content_daily_performance, month={MONTH}")
print("Identifiers will be used only for grain checks and grouping; no raw identifiers are printed.")

Warehouse slice: fact_content_daily_performance, month=2026-03
Identifiers will be used only for grain checks and grouping; no raw identifiers are printed.


## 2. Fields and availability

### Features
The five features are calculated from March 1–15 only: first-half impressions, first-half clicks, first-half click rate, first-half average position, and first-half active impression days.

### Label / proxy
`declined_second_half` is 1 when second-half impressions per day are below 80% of first-half impressions per day. It is an observed movement proxy, not a causal outcome.

### Context
`content_hash_id`, `client_hash_id`, `report_date`, and `ga4_data_available` support grouping, dates, and availability checks. They are not model features.

### Excluded
- `gsc_impressions` and other second-half outcome values: unavailable at the decision moment.
- Any label-derived column, including `declined_second_half`: it is used only to demonstrate leakage, then removed.
- Raw queries, URLs, client names, and tokens: private or not needed for this lane.

**Availability rule:** rows with `ga4_data_available IS TRUE` are counted as available for the warehouse availability check. The five GSC features do not silently convert unavailable analytics rows into evidence.

In [2]:
# Confirm the warehouse objects expose the fields used below without materializing the panel.
fact_schema = con.sql(f"DESCRIBE SELECT * FROM {FACT_MONTH}").df()
required_columns = {
    "client_hash_id", "content_hash_id", "report_date", "gsc_impressions",
    "gsc_clicks", "gsc_avg_position", "ga4_data_available",
}
missing_columns = sorted(required_columns - set(fact_schema["column_name"]))
print(f"Required warehouse fields present: {len(required_columns) - len(missing_columns)}/{len(required_columns)}")
print(f"Missing required fields: {missing_columns}")
assert not missing_columns

print("Contract fields are tied to the daily fact table and the March partition.")

Required warehouse fields present: 7/7
Missing required fields: []
Contract fields are tied to the daily fact table and the March partition.


## 3. Three verification queries on the March 2026 mid-panel slice

Exactly three checks follow. They return aggregate results only:

1. **Grain:** duplicate daily content/client/date keys.
2. **Slice size and span:** row count, distinct content count, and minimum/maximum date.
3. **Availability:** rows surviving `ga4_data_available IS TRUE`.

In [3]:
# Query 1 — grain: one row per client, content item, and report date.
grains = con.sql(f"""
    SELECT COUNT(*) AS duplicate_grains
    FROM (
        SELECT client_hash_id, content_hash_id, report_date
        FROM {FACT_MONTH}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").df()

duplicate_grains = int(grains.loc[0, "duplicate_grains"])
print(f"Duplicate client/content/date grains: {duplicate_grains:,}")
assert duplicate_grains == 0

Duplicate client/content/date grains: 0


In [4]:
# Query 2 — slice size and observed date span.
slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date
    FROM {FACT_MONTH}
""").df()
print(slice_summary.to_string(index=False))
assert int(slice_summary.loc[0, "row_count"]) > 0
assert str(slice_summary.loc[0, "first_report_date"])[:10] == MONTH_START
assert str(slice_summary.loc[0, "last_report_date"])[:10] == MONTH_END

 row_count  content_items first_report_date last_report_date
   9841378         331437        2026-03-01       2026-03-31


In [5]:
# Query 3 — availability: explicitly retain only rows with analytics availability TRUE.
availability = con.sql(f"""
    SELECT COUNT(*) AS rows_with_ga4_available
    FROM {FACT_MONTH}
    WHERE ga4_data_available IS TRUE
""").df()

available_rows = int(availability.loc[0, "rows_with_ga4_available"])
print(f"Rows surviving ga4_data_available IS TRUE: {available_rows:,}")
assert available_rows > 0

Rows surviving ga4_data_available IS TRUE: 413,966


In [7]:
# Build one content-level frame from the March daily slice.
feature_frame = con.sql(f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
            SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_clicks ELSE 0 END) AS first_half_clicks,
            AVG(CASE WHEN report_date <= DATE '{MIDPOINT}' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS first_half_avg_position,
            COUNT(DISTINCT CASE WHEN report_date <= DATE '{MIDPOINT}' AND gsc_impressions > 0 THEN report_date END) AS first_half_active_days,
            SUM(CASE WHEN report_date > DATE '{MIDPOINT}' THEN gsc_impressions ELSE 0 END) AS second_half_impressions,
            COUNT(DISTINCT CASE WHEN report_date > DATE '{MIDPOINT}' THEN report_date END) AS second_half_days,
            COUNT(DISTINCT CASE WHEN report_date <= DATE '{MIDPOINT}' THEN report_date END) AS first_half_days
        FROM {FACT_MONTH}
        GROUP BY 1, 2
    )
    SELECT
        client_hash_id,
        content_hash_id,
        first_half_impressions,
        first_half_clicks,
        100.0 * first_half_clicks / NULLIF(first_half_impressions, 0) AS first_half_ctr,
        COALESCE(first_half_avg_position, 0) AS first_half_avg_position,
        first_half_active_days,
        CAST(
            second_half_impressions / NULLIF(second_half_days, 0)
            < 0.8 * first_half_impressions / NULLIF(first_half_days, 0)
            AS INTEGER
        ) AS declined_second_half
    FROM monthly
    WHERE first_half_impressions > 0
""").df()

feature_columns = [
    "first_half_impressions",
    "first_half_clicks",
    "first_half_ctr",
    "first_half_avg_position",
    "first_half_active_days",
]
assert len(feature_columns) == 5
assert feature_frame[feature_columns].notna().all().all()
assert feature_frame["declined_second_half"].nunique() == 2

print(f"Feature rows: {len(feature_frame):,}")
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")
print(f"Observed second-half decline proxy rate: {feature_frame['declined_second_half'].mean():.3f}")
print(feature_frame[feature_columns].describe().loc[["count", "mean", "50%"]].round(2).to_string())

Feature rows: 151,981
Feature columns (5): ['first_half_impressions', 'first_half_clicks', 'first_half_ctr', 'first_half_avg_position', 'first_half_active_days']
Observed second-half decline proxy rate: 0.359
       first_half_impressions  first_half_clicks  first_half_ctr  first_half_avg_position  first_half_active_days
count               151981.00          151981.00       151981.00                151981.00               151981.00
mean                   838.98               2.53            0.45                    16.40                   10.79
50%                    107.00               0.00            0.00                     8.67                   13.00


## 4. Five features from the same month

The feature frame uses only March 1–15, which is available at the midpoint decision moment. Each feature has an explicit availability explanation:

- `first_half_impressions`: knowable because Search Console has reported impressions through March 15.
- `first_half_clicks`: knowable because Search Console has reported clicks through March 15.
- `first_half_ctr`: knowable because it is calculated from first-half clicks and impressions.
- `first_half_avg_position`: knowable because observed average position is available through March 15.
- `first_half_active_days`: knowable because the daily fact shows which first-half dates had impressions.

The second half is reserved for the observed movement proxy.

## 5. Leakage trap: add it, score it, remove it

To reproduce the lesson from notebook 02, I deliberately add an exact copy of the observed label as `leak_label_copy`. A quick decision-tree score should become nearly perfect because the model has been handed the answer. The honest score is measured again after deleting that column.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

model_data = feature_frame.dropna(subset=feature_columns + ["declined_second_half"]).copy()
train_idx, test_idx = train_test_split(
    model_data.index,
    test_size=0.25,
    random_state=42,
    stratify=model_data["declined_second_half"],
)

leaky = model_data[feature_columns].copy()
leaky["leak_label_copy"] = model_data["declined_second_half"]
honest = model_data[feature_columns].copy()

def quick_score(frame):
    model = DecisionTreeClassifier(max_depth=3, random_state=42)
    model.fit(frame.loc[train_idx], model_data.loc[train_idx, "declined_second_half"])
    return model.score(frame.loc[test_idx], model_data.loc[test_idx, "declined_second_half"])

leaky_score = quick_score(leaky)
honest_score = quick_score(honest)
print(f"Quick score with deliberate label leak: {leaky_score:.3f}")
print(f"Honest quick score after removing the leak: {honest_score:.3f}")
assert leaky_score >= 0.99
assert "leak_label_copy" not in honest.columns
assert honest_score < leaky_score

feature_frame = feature_frame.drop(columns=["declined_second_half"])
print("Leakage column deleted; final feature frame contains only the five decision-time features.")
assert list(feature_frame[feature_columns].columns) == feature_columns

Quick score with deliberate label leak: 1.000
Honest quick score after removing the leak: 0.641
Leakage column deleted; final feature frame contains only the five decision-time features.


## 6. Named limitation

March is a single mid-panel month and the panel is unbalanced across clients. This slice can measure directional, observed associations for the sampled month, but it cannot establish causal refresh effects or guarantee that the signal generalizes to every client or future month.

## Self-check

- [x] Five plain-words contract answers are stated.
- [x] Exactly three aggregate verification queries are included.
- [x] Availability uses `ga4_data_available IS TRUE`.
- [x] Exactly five decision-time features are built from March 1–15.
- [x] The label-derived feature is scored, removed, and excluded from the final frame.
- [x] One named limitation is documented.
- [ ] Run all cells with warehouse access, commit the executed notebook, and submit the repository URL.